# Comandos de la cátedra — cheat sheet

Todo lo que aparece acá está usado en los Colab de `docs/Clases practicas/`.
Los ejemplos usan **los datos del TPE**, así que podés copiarlos y pegar directo.

Todos los ejemplos asumen que ya cargaste el dataset:

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_excel('25-testcar-2026-01-21.xlsx')
```

Y que hiciste el filtrado (esto va PRIMERO, antes de agrupar):

```python
comb = df[~df['Test Fuel Type Description'].isin(['Electricity','Hydrogen 5'])]
comb = comb[comb['Test Category'] == 'FTP'].copy()
comb.loc[comb['RND_ADJ_FE'] > 500, 'RND_ADJ_FE'] = None   # sacar centinelas
```

---

## Práctica 2 — Acceso y análisis de datos

| Comando | Para qué | Ejemplo con tus datos |
|---|---|---|
| `pd.read_csv(f)` | leer CSV | (tu archivo es xlsx, ver nota al final) |
| `pd.read_excel(f)` | leer Excel | `df = pd.read_excel('25-testcar-2026-01-21.xlsx')` |
| `.info()` | ver estructura y nulos | `df.info()` |
| `.describe()` | resumen de las numéricas | `df.describe()` |
| `.head(n)` | primeras filas | `df.head(3)` |
| `.value_counts()` | contar categorías | `df['Test Category'].value_counts()` |
| `.groupby()` | agrupar y resumir | `df.groupby('Test Category')['RND_ADJ_FE'].mean()` |
| `.sort_values(by)` | ordenar | `df.sort_values('Rated Horsepower').head()` |
| `.sort_index()` | ordenar por índice | `df.sort_index()` |
| `.mean()` `.median()` `.max()` `.sum()` | medidas | `df['Rated Horsepower'].mean()` |
| `.astype(t)` | cambiar tipo | `df['Model Year'].astype(str)` |
| `.to_csv(f)` | guardar CSV | `df.head(50).to_csv('salida.csv', index=False)` |
| `.to_excel(f)` | guardar Excel | `df.head(50).to_excel('salida.xlsx', index=False)` |

**Ejemplo completo (el que más vas a usar):**

```python
comb.groupby('Drive System Description')['RND_ADJ_FE'].agg(['count','mean','std'])
```

## Práctica 3 — Correlación

| Comando | Para qué | Ejemplo |
|---|---|---|
| `.corr()` | matriz de correlación | `comb[['RND_ADJ_FE','Rated Horsepower']].corr()` |
| `np.corrcoef(x, y)` | coeficiente puntual | `np.corrcoef(x, y)[0, 1]` |
| `stats.pearsonr(x, y)` | correlación + p-valor | `stats.pearsonr(x, y)` |
| `sns.heatmap(m)` | mapa de calor | `sns.heatmap(m, annot=True)` |

```python
sub = comb.dropna(subset=['Rated Horsepower','RND_ADJ_FE'])
r, p = stats.pearsonr(sub['Rated Horsepower'], sub['RND_ADJ_FE'])
print(r, p)     # -> -0.6928  0.0
```

## Práctica 4 — Limpieza y codificación

| Comando | Para qué | Ejemplo |
|---|---|---|
| `pd.get_dummies(df)` | variables dummies (0/1) | `pd.get_dummies(df[['Drive System Code']])` |
| `pd.concat([a,b])` | pegar tablas | `pd.concat([df, df])` |
| `.replace(a, b)` | cambiar valores | `df['Transmission Lockup?'].replace({'Y':'si','N':'no'})` |
| `.map(dict)` | traducir valores | `df['Test Category'].map({'FTP':1,'HWY':2})` |
| `.apply(f)` | aplicar función a cada valor | `df['Test Category'].apply(lambda x: len(str(x)))` |
| `.isna()` | ver nulos | `df.isna().sum()` |
| `.drop(columns=)` | borrar columnas | `df.drop(columns=['FE_UNIT','Model Year'])` |
| `.drop_duplicates()` | borrar repetidos | `df.drop_duplicates()` |
| `.duplicated()` | marcar repetidos | `df.duplicated().sum()` |
| `.copy()` | copiar | `comb = df.copy()` |

**Reemplazo de `pd.cut` (no existe en la cátedra, se hace con `apply`):**

```python
def tramo(peso):
    if peso < 3500: return 'liviano'
    if peso < 4500: return 'mediano'
    if peso < 5500: return 'pesado'
    return 'muy pesado'

comb['tramo_peso'] = comb['Equivalent Test Weight (lbs.)'].apply(tramo)
comb.groupby('tramo_peso')['RND_ADJ_FE'].mean()
```

## Práctica 5 — Visualización

| Comando | Para qué | Ejemplo |
|---|---|---|
| `plt.hist(x, bins=)` | histograma | `plt.hist(comb['RND_ADJ_FE'].dropna(), bins=40)` |
| `plt.boxplot([...])` | boxplot | `plt.boxplot([g for g in [comb['RND_ADJ_FE'].dropna()]])` |
| `plt.bar(x, y)` | barras | `plt.bar(comb['Drive System Description'].value_counts().index, ...)` |
| `plt.scatter(x, y)` | dispersión | `plt.scatter(sub['Rated Horsepower'], sub['RND_ADJ_FE'])` |
| `plt.pie(v)` | torta | `plt.pie([1,2,3])` |
| `plt.title/ xlabel/ ylabel` | rótulos | `plt.title('Consumo por peso')` |
| `plt.grid()` `plt.legend()` `plt.ylim()` | extras | `plt.grid(True)` |
| `sns.boxplot(data=, x=, y=)` | boxplot por grupo | `sns.boxplot(data=comb, x='Drive System Description', y='RND_ADJ_FE')` |
| `pd.plotting.scatter_matrix(df)` | matriz de dispersión | `pd.plotting.scatter_matrix(comb[['RND_ADJ_FE','Rated Horsepower']].dropna())` |
| `.select_dtypes(include='number')` | solo numéricas | `df.select_dtypes(include='number')` |
| `.isin([...])` | filtrar por lista | `df['Test Category'].isin(['FTP','HWY'])` |

## Práctica 6 — Tests de hipótesis

| Comando | Para qué |
|---|---|
| `stats.ttest_ind(a, b)` | t-test: dos grupos independientes |
| `stats.mannwhitneyu(a, b)` | Mann-Whitney: dos grupos, no paramétrico |
| `stats.kruskal(*grupos)` | Kruskal-Wallis: más de dos grupos, no paramétrico |
| `stats.levene(a, b)` | Levene: verificar homocedasticidad |
| `stats.probplot(x, dist='norm')` | verificar normalidad |
| `sklearn.preprocessing.StandardScaler` | estandarizar para multivariada |

```python
fwd = comb[comb['Drive System Description']=='2-Wheel Drive, Front']['RND_ADJ_FE'].dropna()
awd = comb[comb['Drive System Description']=='All Wheel Drive']['RND_ADJ_FE'].dropna()

print(stats.levene(fwd, awd))                      # homocedasticidad
print(stats.ttest_ind(fwd, awd, equal_var=False)) # t-test
print(stats.mannwhitneyu(fwd, awd))               # alternativa no paramétrica
```

---

## Trampas (las tres que te van a frenar)

**1. `dropna()` en dos columnas por separado rompe `pearsonr`**

```python
# MAL: quedanSeries de distinta longitud
stats.pearsonr(comb['Rated Horsepower'].dropna(), comb['RND_ADJ_FE'].dropna())
# ERROR: x and y must be broadcastable

# BIEN: dropear las dos juntas
sub = comb.dropna(subset=['Rated Horsepower','RND_ADJ_FE'])
```

**2. `probplot` devuelve una tupla de 3**

```python
osm, osr = stats.probplot(sub['RND_ADJ_FE'], dist='norm', plot=None)
# osr = (pendiente, intercepto, r)
```

**3. Asignación encadenada no aplica el cambio (pandas 3)**

```python
mini = pd.DataFrame({'a':[1,2,3,4], 'b':[10,20,30,40]})

mini[mini['a'] > 2]['b'] = 999      # MAL: no aplica (y avisa)
mini.loc[mini['a'] > 2, 'b'] = 999  # BIEN: aplica
```

---

## Nota sobre `pd.read_excel`

**No está en ningún Colab**: en la Práctica 2 se enseña `pd.read_csv` y
`pd.read_table`. Tu archivo es `.xlsx`, así que no hay alternativa. Es el mismo
concepto —leer una tabla y meterla en un DataFrame— con otra extensión. Si te lo
preguntan en la defensa, esa es la respuesta, y conviene declararlo en el informe
como la única desviación del material de clase.

## Lo que NO existe en la cátedra (no lo uses)

| No existe | Usá esto |
|---|---|
| `pd.cut()` | `.apply()` con función propia |
| `pd.crosstab()` | `groupby()` con dos columnas |
| `pd.pivot_table()` | `groupby().mean()` |
| `observed=True` | no hace falta, sobra |
| `pd.read_parquet()` | `pd.read_excel()` |